# Módulo 5: Evaluación de Modelos de Clasificación

## Contenido
1. Accuracy y sus limitaciones
2. Matriz de confusión
3. Precision y Recall
4. Curva ROC y AUC
5. Validación cruzada

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score, confusion_matrix, ConfusionMatrixDisplay,
    precision_score, recall_score, f1_score,
    roc_curve, roc_auc_score
)

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')

In [ ]:
# Reutilizamos el dataset de churn del módulo 4
n = 2000
np.random.seed(42)

df = pd.DataFrame({
    'tenure': np.random.randint(1, 72, n),
    'contract': np.random.choice(['Month-to-month', 'One year', 'Two year'], n, p=[0.5, 0.3, 0.2]),
    'monthly_charges': np.random.uniform(20, 100, n).round(2),
    'internet_service': np.random.choice(['DSL', 'Fiber optic', 'No'], n, p=[0.35, 0.45, 0.2]),
    'online_security': np.random.choice(['Yes', 'No', 'No internet'], n, p=[0.3, 0.5, 0.2]),
    'tech_support': np.random.choice(['Yes', 'No', 'No internet'], n, p=[0.3, 0.5, 0.2]),
    'payment_method': np.random.choice(['Electronic check', 'Mailed check', 'Bank transfer', 'Credit card'], n),
})
df['total_charges'] = (df['monthly_charges'] * df['tenure']).round(2)

prob_churn = (0.1 + 0.3*(df['contract']=='Month-to-month') - 0.15*(df['contract']=='Two year')
              - 0.005*df['tenure'] + 0.003*df['monthly_charges']
              + 0.1*(df['internet_service']=='Fiber optic') - 0.1*(df['online_security']=='Yes')).clip(0.05, 0.95)
df['churn'] = (np.random.random(n) < prob_churn).astype(int)

# Split y preparar
df_train_full, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_train_full, test_size=0.25, random_state=42)

y_train = df_train.pop('churn').values
y_val = df_val.pop('churn').values
y_test = df_test.pop('churn').values

dv = DictVectorizer(sparse=False)
X_train = dv.fit_transform(df_train.to_dict(orient='records'))
X_val = dv.transform(df_val.to_dict(orient='records'))

# Entrenar modelo
modelo = LogisticRegression(solver='liblinear', max_iter=1000)
modelo.fit(X_train, y_train)
y_pred_proba = modelo.predict_proba(X_val)[:, 1]
y_pred = (y_pred_proba >= 0.5).astype(int)

print(f"Modelo entrenado. AUC en val: {roc_auc_score(y_val, y_pred_proba):.4f}")

## 1. Accuracy y sus limitaciones

In [ ]:
acc = accuracy_score(y_val, y_pred)
print(f"Accuracy del modelo: {acc:.4f}")

# Modelo dummy
y_pred_dummy = np.zeros(len(y_val))  # siempre predice "no churn"
acc_dummy = accuracy_score(y_val, y_pred_dummy)
print(f"Accuracy dummy (siempre 0): {acc_dummy:.4f}")
print(f"\nSi la tasa de churn es {y_val.mean():.2%}, un modelo que siempre dice 'no churn'")
print(f"ya tiene {1-y_val.mean():.2%} de accuracy. ¡No es informativo!")

## 2. Matriz de confusión

In [ ]:
cm = confusion_matrix(y_val, y_pred)
tn, fp, fn, tp = cm.ravel()

print(f"True Negatives:  {tn} (correctamente identificados como no churn)")
print(f"False Positives: {fp} (marcados como churn incorrectamente)")
print(f"False Negatives: {fn} (churners NO detectados)")
print(f"True Positives:  {tp} (churners detectados correctamente)")

# Visualizar
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(
    y_val, y_pred,
    display_labels=['No Churn', 'Churn'],
    cmap='Blues', ax=ax
)
plt.title('Matriz de Confusión')
plt.show()

## 3. Precision y Recall

In [ ]:
precision = precision_score(y_val, y_pred)
recall = recall_score(y_val, y_pred)
f1 = f1_score(y_val, y_pred)

print(f"Precision: {precision:.4f} (de los que dije churn, ¿cuántos lo eran?)")
print(f"Recall:    {recall:.4f} (de los que eran churn, ¿cuántos detecté?)")
print(f"F1-score:  {f1:.4f} (balance entre ambas)")

In [ ]:
# Trade-off Precision vs Recall según umbral
umbrales = np.arange(0.1, 0.9, 0.05)
precisions, recalls = [], []

for t in umbrales:
    y_t = (y_pred_proba >= t).astype(int)
    precisions.append(precision_score(y_val, y_t, zero_division=0))
    recalls.append(recall_score(y_val, y_t))

plt.figure(figsize=(8, 5))
plt.plot(umbrales, precisions, 'b-', label='Precision')
plt.plot(umbrales, recalls, 'r-', label='Recall')
plt.axvline(x=0.5, color='gray', linestyle='--', alpha=0.5, label='Umbral=0.5')
plt.xlabel('Umbral')
plt.ylabel('Score')
plt.title('Precision vs Recall según umbral')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 4. Curva ROC y AUC

In [ ]:
fpr, tpr, _ = roc_curve(y_val, y_pred_proba)
auc = roc_auc_score(y_val, y_pred_proba)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, 'b-', lw=2, label=f'Logística (AUC={auc:.3f})')
plt.plot([0, 1], [0, 1], 'k--', label='Aleatorio (AUC=0.5)')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate (Recall)')
plt.title('Curva ROC')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"AUC: {auc:.4f}")

In [ ]:
# Comparar múltiples modelos
modelos = {
    'Logística': LogisticRegression(solver='liblinear', max_iter=1000),
    'Árbol (d=5)': DecisionTreeClassifier(max_depth=5, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42),
}

plt.figure(figsize=(8, 6))
for nombre, mod in modelos.items():
    mod.fit(X_train, y_train)
    y_proba = mod.predict_proba(X_val)[:, 1]
    fpr, tpr, _ = roc_curve(y_val, y_proba)
    auc = roc_auc_score(y_val, y_proba)
    plt.plot(fpr, tpr, lw=2, label=f'{nombre} (AUC={auc:.3f})')

plt.plot([0, 1], [0, 1], 'k--', label='Aleatorio')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Comparación de Modelos - ROC')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 5. Validación Cruzada

In [ ]:
# Preparar datos completos para CV
X_full = dv.transform(df_train_full.drop(columns=['churn'], errors='ignore').to_dict(orient='records'))
y_full = df_train_full['churn'].values if 'churn' in df_train_full.columns else np.concatenate([y_train, y_val])

# Si ya quitamos churn antes, reconstruir
X_full = np.vstack([X_train, X_val])
y_full = np.concatenate([y_train, y_val])

print(f"{'Modelo':<20} {'AUC medio':>10} {'± Std':>8}")
print("-" * 40)

for nombre, mod in modelos.items():
    scores = cross_val_score(mod, X_full, y_full, cv=5, scoring='roc_auc')
    print(f"{nombre:<20} {scores.mean():>10.4f} {scores.std():>8.4f}")

In [ ]:
# Resumen final
print("\n" + "=" * 50)
print("RESUMEN DE EVALUACIÓN")
print("=" * 50)
print(f"\nMejor modelo: Random Forest")
print(f"\nMétricas en validación (umbral=0.5):")

rf = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
rf.fit(X_train, y_train)
y_rf_proba = rf.predict_proba(X_val)[:, 1]
y_rf_pred = (y_rf_proba >= 0.5).astype(int)

print(f"  AUC:       {roc_auc_score(y_val, y_rf_proba):.4f}")
print(f"  Accuracy:  {accuracy_score(y_val, y_rf_pred):.4f}")
print(f"  Precision: {precision_score(y_val, y_rf_pred):.4f}")
print(f"  Recall:    {recall_score(y_val, y_rf_pred):.4f}")
print(f"  F1:        {f1_score(y_val, y_rf_pred):.4f}")